# Determine how location impacts interest rates

This notebook provides a Python companion to Lauren Scott Griffin's ArcGIS Pro tutorial [Determine how location impacts interest rates](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/). It follows the original tutorial step by step, with modifications for replicability and synthesis with broader `arcpy` workflows. 

By the end of this tutorial, you should be able to:
1) Analyze hot spots using Getis-Ord Gi*.
2) Build and interpret a linear regression model. 
3) Build and interpret a geographically weighted regression model.

## Getting started
This was designed to be run as a Jupyter Notebook inside of ArcGIS Pro. To replicate, download the [Online Lending Data](https://www.arcgis.com/sharing/rest/content/items/341ce3d6bb434f75b7fe971893091ed1/data), which includes a `.aprx` project file and data in a `.gdb` database.  Once you open that project file, also download this notebook add it to your project using `Insert > Add and Open Notebook`. It will now be visible in a newly visible folder called `Notebooks` in the catalog pane. 

In [43]:
# Now load the relevant libraries
import arcpy  # for ArcGIS tools
import os  # for interacting with your local file directory
import pandas  # for managing tabular data


By default, a notebook in ArcGIS Pro will define your working directory as whatever folder the .aprx or .ppkx file is found in. If you want to introduce other data, an easy way to do that is to simply copy/paste all files into the default folder.

In [2]:
# and set up your local working environment
aprx = arcpy.mp.ArcGISProject("Current") # your current aprx file
default_gdb = aprx.defaultGeodatabase  # the default geodatabase of the aprx
default_folder = aprx.homeFolder  # the default folder of the aprx
arcpy.env.overwriteOutput = True  # allows ArcGIS to overwrite files
map = aprx.listMaps()[0] # specify which map you're using 
crs = map.spatialReference  # borrow the default projection from that map

print("Directory: " + default_folder)
print("Geodatabase: " + default_gdb)
print("Activated map: " + map.name)
print("Coordinate reference system: " + crs.name)

Directory: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Teaching\INFO 612 Advanced GIS\Lessons\08_geographic_regression\online-lending-data
Geodatabase: C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Teaching\INFO 612 Advanced GIS\Lessons\08_geographic_regression\online-lending-data\OnlineLending.gdb
Activated map: Map
Coordinate reference system: USA_Contiguous_Albers_Equal_Area_Conic


## [Create a hot spot map](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#create-a-hot-spot-map)
The first part of the tutorial sets up and then interprets a hot spot analysis of mortgage interest rates using the Getis-Ord Gi* metric. 

#### [Select tracts with at least 30 loans](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#:~:text=Select%20tracts%20with%20at%20least%2030%20loans)

In [3]:
# select high frequency areas
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view="ZIP3 Loan Data",
    selection_type="NEW_SELECTION",
    where_clause="AcceptedLoans >= 30",
    invert_where_clause=None
)

<Result 'ZIP3 Loan Data'>

In [4]:
# and save a copy
arcpy.management.CopyFeatures(
    in_features="ZIP3 Loan Data",
    out_feature_class=os.path.join(default_gdb, "ZIP3LoanData_Analysis_Data"),
    config_keyword="",
    spatial_grid_1=None,
    spatial_grid_2=None,
    spatial_grid_3=None
)

# verify it's in the geodatabase
arcpy.ListFeatureClasses()

['ZIP3LoanData', 'StateBoundaries', 'ZIP3LoanData_Analysis_Data']

#### [Analyze interest rate hot spots](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#:~:text=Analyze%20interest%20rate%20hot%20spots)

In [5]:
# hot spot analysis
arcpy.stats.HotSpots(
    Input_Feature_Class="ZIP3LoanData_Analysis_Data",
    Input_Field="AveInterestRate",
    Output_Feature_Class=os.path.join(default_gdb, "Interest_Rate_Hot_Spots"),
    Conceptualization_of_Spatial_Relationships="FIXED_DISTANCE_BAND",
    Distance_Method="EUCLIDEAN_DISTANCE",
    Standardization="ROW",
    Distance_Band_or_Threshold_Distance=None,
    Self_Potential_Field=None,
    Weights_Matrix_File=None,
    Apply_False_Discovery_Rate__FDR__Correction="APPLY_FDR",
    number_of_neighbors=None
)

<Result 'C:\\Users\\johnl\\My Drive (jlauerma@pratt.edu)\\Teaching\\INFO 612 Advanced GIS\\Lessons\\08_geographic_regression\\online-lending-data\\OnlineLending.gdb\\Interest_Rate_Hot_Spots'>

In [48]:
# and just view to verify
# pull into a geodata frame
gdf = GeoAccessor.from_featureclass(f"{default_gdb}\\{"Interest_Rate_Hot_Spots"}")
gdf

,OBJECTID,SOURCE_ID,AveInterestRate,GiZScore,GiPValue,NNeighbors,Gi_Bin,SHAPE
0,1,1,13.209079,-1.482349,0.138247,27,0,"{""rings"": [[[-2035312.2664130032, -135762.8630..."
1,2,2,13.175083,-1.482349,0.138247,27,0,"{""rings"": [[[-2038945.9897855967, -167928.9845..."
2,3,3,14.047677,-1.482349,0.138247,27,0,"{""rings"": [[[-2034234.677861821, -159523.84009..."
3,4,4,12.296141,-1.482349,0.138247,27,0,"{""rings"": [[[-2039721.3290224224, -148588.4646..."
4,5,5,13.222388,-1.273769,0.202745,28,0,"{""rings"": [[[-2028654.462696774, -170037.72499..."
...,...,...,...,...,...,...,...,...
803,804,804,13.565878,0.852837,0.39375,8,0,"{""rings"": [[[-1947180.430726966, 1443854.49706..."
804,805,805,14.031608,0.97246,0.330822,8,0,"{""rings"": [[[-2004341.007541947, 1355489.65446..."
805,806,806,14.008914,0.359218,0.719432,10,0,"{""rings"": [[[-1982846.2417210005, 1373664.9906..."
806,807,807,13.928453,0.463246,0.643188,9,0,"{""rings"": [[[-2116030.7770469263, 1358527.7049..."


## [Create a regression model](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#create-a-regression-model)
This section explores predictors of interest rates using linear and geographically weighted regression models. 

#### [Perform regression analysis](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#create-a-regression-model:~:text=Perform%20regression%20analysis)

We'll start with a simple linear regression. This makes no assumptions about spatial structure: it just tests whether one X variable predicts behavior in the Y variable. Once you get results, look for:
- Summary table metrics such as the coefficient (interpretation: a one unit increase in X predicts this many units increase in Y) and p value (interpretation: smaller means more statistically significant)
- Diagnostics table shows overall model performance. For now, just look at the multiple R2 (interpretation: the model explains __ % of total variance in Y).
- Scatter plot of predicted versus actual values (interpretation: tighter clustering around the line indicates a more accurate model)
- Histogram of residuals (interpretation: should be a roughly normal distribution. If not, your data do not meet assumptions of a regression model).
- Residuals vs. predicted plot (interpretation: should be clustered with few outliers. If not, you have assumption violations.)

In [8]:
# simple linear regression
arcpy.stats.GeneralizedLinearRegression(
    in_features="ZIP3LoanData_Analysis_Data",
    dependent_variable="AveInterestRate",
    model_type="CONTINUOUS",
    output_features=r"C:\Users\johnl\My Drive (jlauerma@pratt.edu)\Teaching\INFO 612 Advanced GIS\Lessons\08_geographic_regression\online-lending-data\OnlineLending.gdb\Average_Interest_Rates_vs_Loan_Grades",
    explanatory_variables="AveLoanGrade",
    distance_features=None,
    prediction_locations=None,
    explanatory_variables_to_match=None,
    explanatory_distance_matching=None,
    output_predicted_features=None,
    output_trained_model=None
)

<Result 'C:\\Users\\johnl\\My Drive (jlauerma@pratt.edu)\\Teaching\\INFO 612 Advanced GIS\\Lessons\\08_geographic_regression\\online-lending-data\\OnlineLending.gdb\\Average_Interest_Rates_vs_Loan_Grades'>

## [Map correlation variations](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#map-correlation-variations)
This final subsection explores underlying spatial patterns, models them in a geographically weighted regression (using both a KNN and distance band spatial weight), and maps the results. 

#### [Find the minimum neighbor distance](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#map-correlation-variations:~:text=Find%20the%20minimum%20neighbor%20distance)

In [49]:
# find distance range of k=10 neighbors
arcpy.stats.CalculateDistanceBand(
    Input_Features="ZIP3LoanData_Analysis_Data",
    Neighbors=10,
    Distance_Method="EUCLIDEAN_DISTANCE"
)

<Result '17802.5732142922'>

In [50]:
# find distance range of k=50 neighbors
arcpy.stats.CalculateDistanceBand(
    Input_Features="ZIP3LoanData_Analysis_Data",
    Neighbors=50,
    Distance_Method="EUCLIDEAN_DISTANCE"
)

<Result '119354.94068387'>

#### [Build the spatial regression model](https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#map-correlation-variations:~:text=Build%20the%20spatial%20regression%20model)

Now we'll build a geographically weighted regression. The primary difference is that we introduce a weighting system into the model, to account for spatial autocorrelation. Interpretation of outputs is largely the same as the model above. But:
- Model diagnostics: Look for increases in adjusted R2
- Manual interval results: The AIC metric tells you the overall efficiency of the model. Lower AIC indicates a more efficient model. As you can see from the table of AIC values by number of neighbors, the optimal value of neighbors would be around 30. 

In [51]:
# GWR with k=10 neighbors
arcpy.stats.GWR(
    in_features="ZIP3LoanData_Analysis_Data",
    dependent_variable="AveInterestRate",
    model_type="CONTINUOUS",
    explanatory_variables="AveLoanGrade",
    output_features=os.path.join(default_gdb, "GWR_Average_Interest_Rate_vs_Average_Loan_Grade"),
    neighborhood_type="NUMBER_OF_NEIGHBORS",
    neighborhood_selection_method="MANUAL_INTERVALS",
    minimum_number_of_neighbors=10,
    maximum_number_of_neighbors=None,
    minimum_search_distance=None,
    maximum_search_distance=None,
    number_of_neighbors_increment=4,
    search_distance_increment=None,
    number_of_increments=11,
    number_of_neighbors=None,
    distance_band=None,
    prediction_locations=None,
    explanatory_variables_to_match=None,
    output_predicted_features=None,
    robust_prediction="ROBUST",
    local_weighting_scheme="BISQUARE",
    coefficient_raster_workspace=None,
    scale=None
)

<Result 'C:\\Users\\johnl\\My Drive (jlauerma@pratt.edu)\\Teaching\\INFO 612 Advanced GIS\\Lessons\\08_geographic_regression\\online-lending-data\\OnlineLending.gdb\\GWR_Average_Interest_Rate_vs_Average_Loan_Grade'>

In [52]:
# GWR with a distance band
arcpy.stats.GWR(
    in_features="ZIP3LoanData_Analysis_Data",
    dependent_variable="AveInterestRate",
    model_type="CONTINUOUS",
    explanatory_variables="AveLoanGrade",
    output_features=os.path.join(default_gdb, "GWR_Average_Interest_Rate_vs_Average_Loan_Grade"),
    neighborhood_type="DISTANCE_BAND",
    neighborhood_selection_method="MANUAL_INTERVALS",
    minimum_number_of_neighbors=None,
    maximum_number_of_neighbors=None,
    minimum_search_distance="400000 Meters",
    maximum_search_distance=None,
    number_of_neighbors_increment=None,
    search_distance_increment="100000 Meters",
    number_of_increments=8,
    number_of_neighbors=None,
    distance_band=None,
    prediction_locations=None,
    explanatory_variables_to_match=None,
    output_predicted_features=None,
    robust_prediction="ROBUST",
    local_weighting_scheme="BISQUARE",
    coefficient_raster_workspace=None,
    scale=None
)

<Result 'C:\\Users\\johnl\\My Drive (jlauerma@pratt.edu)\\Teaching\\INFO 612 Advanced GIS\\Lessons\\08_geographic_regression\\online-lending-data\\OnlineLending.gdb\\GWR_Average_Interest_Rate_vs_Average_Loan_Grade'>

## Additional resources
Scott Griffin, Lauren (2025) Determine how location impacts interest rates. _ArcGIS Tutorial Library_. https://learn.arcgis.com/en/projects/determine-how-location-impacts-interest-rates/#map-correlation-variations